<a href="https://colab.research.google.com/github/ChaitanyaTumu/Requirement-Testcase-Matching/blob/main/LoRA_Finetuned_on_Requirements_TC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This project utilizes Natural Language Processing (NLP) to automate the traceability between software requirements and their corresponding test cases.

In [ ]:
 !pip install transformers datasets accelerate peft bitsandbytes


###  Loading and Merging the Datasets
We load our proprietary JSON data containing Requirements and Test Cases. We then use Pandas to perform an inner join on `Test_case_ID` to pair the specific requirements with their designated test cases.

In [ ]:
import json
import pandas as pd

requirement_path="/content/Requirement_L2H0090.json"
tc_path="/content/Test_case_L2H0090.json"

with open(requirement_path, "r") as f:
    req_data = json.load(f)

with open(tc_path, "r") as f:
    tc_data = json.load(f)

# 1b. Convert to DataFrame
req_df = pd.DataFrame(req_data)
tc_df  = pd.DataFrame(tc_data)

# Quick sanity check
print("Requirements shape:", req_df.shape)
print("Test-cases   shape:", tc_df.shape)
req_df.head()

### Feature Engineering and Negative Sampling
To feed data into BERT, we must combine our text features into a single cohesive string. We concatenate the Requirement text with the Test Case details, preconditions, and postconditions.

Crucially, to teach the model what a "bad match" looks like, we perform **Negative Sampling**. For every correct Requirement-TestCase pair (Label = 1), we randomly select an incorrect Test Case to pair with the Requirement (Label = 0).

In [ ]:
req_df["Test_case_ID"] = req_df["Test_case_ID"].astype(str)
tc_df["Test_case_ID"]  = tc_df["Test_case_ID"].astype(str)

merged = pd.merge(
    left=req_df,
    right=tc_df,
    on="Test_case_ID",
    how="inner",
    suffixes=("_req", "_tc")
)

print("Merged pairs:", merged.shape)
merged.columns



In [ ]:
def safe(col):
    return merged[col].fillna("").astype(str)

merged["text"] = (
    safe("Req_Text") + "\n\n"
  + safe("Test_case_Details") + "\n"
  + safe("Precondition")      + "\n"
  + safe("Postcondition")
)

# Keep only what we need for training:
dataset = merged[[
    "Req_ID_req",           # (optional) you can keep for tracing
    "Test_case_ID",     # same
    "text"
]].copy()
dataset["label"] = 1     # these are _positive_ matches
dataset.head()

In [ ]:
import random

all_tc_ids = tc_df["Test_case_ID"].unique().tolist()
neg_samples = []

for _, row in merged.iterrows():
    req_id = row["Req_ID_req"]
    req_txt = row["Req_Text"]
    # pick one random TC that isn’t the correct one
    wrong = random.choice([tc for tc in all_tc_ids if tc != row["Test_case_ID"]])
    tc_row = tc_df[tc_df["Test_case_ID"] == wrong].iloc[0]

    neg_samples.append({
        "Req_ID":        req_id,
        "Test_case_ID":  wrong,
        "text":          req_txt + "\n\n"
                          + str(tc_row["Test_case_Details"]) + "\n"
                          + str(tc_row["Precondition"]) + "\n"
                          + str(tc_row["Postcondition"]),
        "label": 0
    })

neg_df = pd.DataFrame(neg_samples)
full_df = pd.concat([dataset, neg_df], ignore_index=True).sample(frac=1).reset_index(drop=True)

print("Total examples:", full_df.shape)
full_df.label.value_counts()


###  Splitting and Converting to Hugging Face Datasets
We use `scikit-learn` to split our balanced dataset into 90% training data and 10% validation data. We then convert the Pandas DataFrames into Hugging Face `Dataset` objects, which are optimized for transformer training.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    full_df,
    test_size=0.1,
    stratify=full_df["label"],
    random_state=42
)

print("Train:", train_df.shape, "Val:", val_df.shape)


In [ ]:
from datasets import Dataset

hf_train = Dataset.from_pandas(train_df)
hf_val   = Dataset.from_pandas(val_df)
raw_ds   = {"train": hf_train, "validation": hf_val}


### Tokenization
Machine learning models cannot read raw text; they require numbers. We use the `AutoTokenizer` for `bert-base-uncased` to convert our text pairs into tokens. We apply padding and truncation to ensure all sequences have a uniform length of 512 tokens.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_fn(ex):
    return tokenizer(
        ex["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized = {
    split: ds.map(tokenize_fn, batched=True, remove_columns=["Req_ID","Test_case_ID","text","label"])
    for split, ds in raw_ds.items()
}
# keep labels around
train_labels = train_df["label"].tolist()
val_labels   = val_df["label"].tolist()


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

# 9a. Load base BERT
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2
)

# 9b. Configure & apply LoRA
peft_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    inference_mode=False,
    r=16,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="all",
    target_modules=["key", "query", "value"],
    modules_to_save=["classifier"],
)
model = get_peft_model(model, peft_config)


###  Configuring LoRA (Low-Rank Adaptation)
This is the core of the parameter-efficient fine-tuning.
1. We load the base BERT model configured for 2 labels (Match vs. No-Match).
2. We apply the `LoraConfig`. Instead of updating all 110 million parameters of BERT, LoRA only updates a tiny fraction (the adapter weights), saving massive amounts of GPU memory and time.
3. We set up the `TrainingArguments`, utilizing `fp16` (mixed precision) for faster processing.

In [ ]:
training_args = TrainingArguments(
    output_dir="lora_finetuned_bert",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=2e-4,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"].add_column("labels", train_labels),
    eval_dataset= tokenized["validation"].add_column("labels", val_labels),
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()


In [ ]:
metrics = trainer.evaluate()
print(" Final metrics:", metrics)


In [ ]:
!ls -R /content/lora_finetuned_bert

###  Model Training and Evaluation
We execute the training loop. After training is complete, we evaluate the model against the validation set to obtain our final loss and accuracy metrics, and verify the adapter weights were saved successfully.

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# 1) Load tokenizer and base‐model + LoRA adapter
MODEL_ID = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# load the base sequence‐classification head
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2
)
# wrap it with your LoRA weights
model = PeftModel.from_pretrained(base_model, "/content/lora_finetuned_bert/checkpoint-210")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

###  Inference Pipeline: Loading the Fine-Tuned Model
To use the model in a real-world scenario, we load the base BERT model and attach our newly trained LoRA weights using `PeftModel.from_pretrained()`.

We also define a helper function (`predict_best_testcase`) that takes a brand-new Requirement string, pairs it with every available Test Case in the database, and asks the model to score the probability of a match.

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# 2) Load your candidate test‐cases
tc_path = "/content/Test_case_L2H0090.json"
tc_df   = pd.read_json(tc_path, dtype=str)

# 3) Define a helper that builds the text for each (req,tc) pair
def build_pair_text(req_text, tc_row):
    return (
        req_text.strip() + "\n\n"
      + str(tc_row["Test_case_Details"]).strip() + "\n"
      + str(tc_row["Precondition"]).strip()      + "\n"
      + str(tc_row["Postcondition"]).strip()
    )

# 4) Prediction function
def predict_best_testcase(requirement: str, candidates: pd.DataFrame, top_k=1, batch_size=32):
    # assemble all inputs in batches
    all_scores = []
    for i in range(0, len(candidates), batch_size):
        batch_candidates = candidates.iloc[i : i + batch_size]
        texts = [ build_pair_text(requirement, row)
                  for _, row in batch_candidates.iterrows() ]
        # tokenize in batch
        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        # forward
        with torch.no_grad():
            logits = model(**inputs).logits        # shape (N,2)
            probs  = torch.softmax(logits, dim=-1) # each row = [P(neg), P(pos)]
            pos_scores = probs[:,1]                # P(match)
            all_scores.extend(pos_scores.cpu().tolist())

    # pick top-k from all scores
    scored_candidates = list(zip(candidates["Test_case_ID"].tolist(), all_scores))
    scored_candidates.sort(key=lambda x: x[1], reverse=True)

    # return list of (Test_case_ID, score)
    return scored_candidates[:top_k]

# 5) Try it out


In [ ]:
new_req = "Hazard_Detection system shall detect all obstacles larger than 5cm in the vehicle’s forward path within 50ms."
best = predict_best_testcase(new_req, tc_df, top_k=3)

print("Top 3 matches:")
for tc_id, score in best:
    print(f" • Test_case_ID={tc_id}    score={score:.4f}")

In [ ]:

new_req = "The Lane_Detection_Algorithms module shall send LDA_FC1_IntPXXRole with the value init (0x07)."
best = predict_best_testcase(new_req, tc_df, top_k=3)

print("Top 3 matches:")
for tc_id, score in best:
    print(f" • Test_case_ID={tc_id}    score={score:.4f}")

In [ ]:

new_req = "The system shall store the spare part number in one unique memory location used by both bootloader and application software."
best = predict_best_testcase(new_req, tc_df, top_k=3)

print("Top 3 matches:")
for tc_id, score in best:
    print(f" • Test_case_ID={tc_id}    score={score:.4f}")